## 生成数据点

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
### 内部点 
def get_domain_points(Nf):
    x = torch.rand(Nf, 1) * 2 * np.pi  # x ∈ [0, 2π]
    t = torch.rand(Nf, 1)              # t ∈ [0, 1]
    x.requires_grad_(True)
    t.requires_grad_(True)
    return x, t

### 边界点 
def get_boundary_points(Nb):
    # x=0 和 x=2π 处的点
    x_bc = torch.zeros((Nb//2, 1)).requires_grad_(True)  # x=0
    x_bc2 = torch.full((Nb//2, 1), 2*np.pi).requires_grad_(True)  # x=2π
    t_bc = torch.rand(Nb//2, 1).requires_grad_(True)
    t_bc2 = torch.rand(Nb//2, 1).requires_grad_(True)
    return torch.cat([x_bc, x_bc2]), torch.cat([t_bc, t_bc2]) ##前半段是x=0，后半段是x=2Π

### 初始点
def get_initial_points(Ni):
    x_ic = torch.linspace(0, 2*np.pi, Ni).reshape(-1, 1)
    t_ic = torch.zeros(Ni, 1)  # 明确形状
    x_ic.requires_grad_(True)
    t_ic.requires_grad_(True)
    return x_ic, t_ic

## 定义损失函数

In [ ]:
def compute_loss(model, x_f, t_f, x_bc, t_bc, x_ic, t_ic):
    # PDE 残差
    u_f = model(x_f, t_f)
    u_t = torch.autograd.grad(u_f.sum(), t_f, create_graph=True)[0]
    u_x = torch.autograd.grad(u_f.sum(), x_f, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x.sum(), x_f, create_graph=True)[0]
    pde_residual = u_t - u_xx
    loss_pde = torch.mean(pde_residual**2)

    # 边界条件
    u_bc = model(x_bc, t_bc)
    loss_bc = torch.mean(u_bc**2)

    # 初始条件
    u_ic = model(x_ic, t_ic)
    u_exact_ic = -torch.cos(x_ic) + 1
    loss_ic = torch.mean((u_ic - u_exact_ic)**2)

    return loss_pde + 0.1 * loss_bc + 0.1 * loss_ic

## 定义神经网络

In [ ]:
class PINN(nn.Module):
    def __init__(self, layers, activation=nn.Tanh()):
        super(PINN, self).__init__()
        modules = []
        for i in range(len(layers) - 2):
            modules.append(nn.Linear(layers[i], layers[i+1]))
            modules.append(activation)  # 每层后加激活
        modules.append(nn.Linear(layers[-2], layers[-1]))  # 输出层无激活
        self.net = nn.Sequential(*modules)
    
    def forward(self, x, t):
        input = torch.cat([x, t], dim=1)
        return self.net(input)

## 训练模型

In [ ]:
model = PINN([2, 64, 64, 64, 64, 1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 数据点数量
Nf = 10000
Nb = 100
Ni = 100

x_f, t_f = get_domain_points(Nf)
x_bc, t_bc = get_boundary_points(Nb)
x_ic, t_ic = get_initial_points(Ni)

x_f = x_f.to(device)
t_f = t_f.to(device)
x_bc = x_bc.to(device)
t_bc = t_bc.to(device)
x_ic = x_ic.to(device)
t_ic = t_ic.to(device)

# 训练
epochs = 100
losses = []

for epoch in range(epochs):
    optimizer.zero_grad()
    loss = compute_loss(model, x_f, t_f, x_bc, t_bc, x_ic, t_ic)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
    losses.append(loss.item())

## 绘图

In [ ]:
## 绘制等高线图
num = 100
x = np.linspace(0, 2*np.pi, num)
t = np.linspace(0,1, num )
X,T = np.meshgrid(x, t)

X_to_tensor = torch.tensor(X, dtype=torch.float32).reshape(-1,1).to(device)
T_to_tensor = torch.tensor(T, dtype=torch.float32).reshape(-1,1).to(device)
with torch.no_grad():
    Z_to_tensor = model(X_to_tensor,T_to_tensor)
Z = Z_to_tensor.cpu().numpy().reshape(X.shape)

plt.figure(figsize=(8,6))
plt.contour(X,T,Z, levels=50)
plt.show()

In [ ]:
## 绘制初始点函数对比图 
x_init = np.linspace(0, 2*np.pi, 100)
u_exact_init = -np.cos(x_init) + 1
plt.figure(figsize=(8,6))
plt.plot(x_init,u_exact_init, label='Theoretical Initial Condition', linestyle='-')
with torch.no_grad():
    u_pred_init = model(torch.tensor(x_init, dtype=torch.float32).reshape(-1,1).to(device),torch.zeros((100,1)).to(device))
u_pred_init = u_pred_init.cpu().numpy().reshape(x_init.shape)
plt.plot(x_init,u_pred_init, label='Model Prediction at t=0', linestyle='--')
plt.show()